#  Why do we need prompt versioning?
Suppose your sentiment classifier has this prompt:
Version 1.0
Classify the sentiment as POSITIVE,
NEGATIVE, or NEUTRAL.


You improve it by adding definitions.
Version 1.1
Classify sentiment using these definitions:
POSITIVE = praise or satisfaction
NEGATIVE = criticism or dissatisfaction
NEUTRAL = factual text without clear sentiment

Return only one label.


Later, you discover that Version 1.0 performed better on a particular dataset.
Without versioning, you may not know which prompt was used, or how to restore the earlier version.
2. Architecture
Prompt repository
Store prompt text and version metadata

Select prompt version
For example, sentiment_v1.1

Run LLM + evaluation
Record accuracy, latency, and token usage

Version history
Trace results back to the exact prompt


3. Build a prompt registry
We'll store prompt versions in a Python dictionary for now. Later, we could move this registry to a JSON file or database.

In [12]:
from dotenv import load_dotenv
load_dotenv()
import os

groq_api_key = os.getenv("GROQ_API_KEY")

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"])

/Users/vimal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [8]:
from datetime import datetime

prompt_registry = { 
    "sentiment_v1.0" : {
        "version" : "1.0",
        "created_at" : "2026-10-11",
        "description" : "Baseic Sentiment Classficiation",
        "template" : """ Classify the sentiment as Positive, Negative or Neutral
        Return only the label. 

        Text : {text} 
        """},

            "sentiment_v1.1": {
        "version": "1.1",
        "created_at": "2026-10-11",
        "description": "Classification with label definitions",
        "template": """
Classify the sentiment using these definitions:
- POSITIVE: praise or satisfaction
- NEGATIVE: criticism or dissatisfaction
- NEUTRAL: factual text without clear sentiment

Return exactly one label: POSITIVE, NEGATIVE, or NEUTRAL.

Text: {text}
"""
} }

In [11]:
def run_prompt(version_id, text):
    if version_id not in prompt_registry:
        raise ValueError(f"Unknown prompt version : {version_id}")
    prompt_data = prompt_registry[version_id]

    prompt = prompt_data["template"].format(text = text)

    response = llm.invoke(prompt)

    return { "prompt_id" : version_id,
                        "prompt_version": prompt_data["version"],
                        "text": text,
                        "response": response.content.strip(),
                        "total_tokens" : response.usage_metadata}

result = run_prompt(
    "sentiment_v1.1",
    "The application is easy to use and works perfectly."
)

print(result)

{'prompt_id': 'sentiment_v1.1', 'prompt_version': '1.1', 'text': 'The application is easy to use and works perfectly.', 'response': 'POSITIVE', 'total_tokens': {'input_tokens': 135, 'output_tokens': 89, 'total_tokens': 224, 'input_token_details': {}, 'output_token_details': {'reasoning': 78}}}


You can now see how different prompt versions behave on the same input. For a reliable comparison, run both against the same labeled evaluation dataset, as we did in Chapter 29.